**RAG as a sequential pipeline**

I outlined the complete lifecycle of an advanced RAG (Retrieval-Augmented Generation) Pipeline  

1. Preprocessing

2. Index Construction

3. LLM-Augmented Retrieval

4. Evaluation

**Evaluating the final output**

I outlined the fundamental pillars to evaluate the results

4.1 Retrieval Quality

4.2 Generation Quality

4.3 End-to-End QA Accuracy

In [ ]:
test_data = []

with open("/kaggle/input/datasets/saridacharoenngampit/test-data/test.jsonl", "r") as f:
    for line in f:
        test_data.append(json.loads(line))

**4.1 Retrieval Quality**

***Expected Results when running retrieval evaluation***

**Input**

In [ ]:
'question_i'

**Output**

In [ ]:
'document_i'

***4.1.1 Recall@5***

**4.1.1 Baseline LLM**

In [ ]:
from sklearn.metrics import ndcg_score
import numpy as np

def recall_at_k(ranked, gold, k=5):
    return int(gold in ranked[:k])

recall_scores = []

for item in test_data:
    gold   = item["answer"]
    ranked = list(item["options"].values())
    recall_scores.append(recall_at_k(ranked, gold, 5))

print("Total Samples :", len(recall_scores))
print("Mean Recall@5 :", round(sum(recall_scores) / len(recall_scores), 4))

Total Samples : 100
Mean Recall@5 : 0.4200


**4.1.2 Advanced RAG**

In [ ]:
adv_recall_scores = []

for _, item in df.iterrows():
    gold   = item["gold"]
    ranked = item["ranked_docs"]
    if not isinstance(ranked, list) or len(ranked) == 0:
        continue
    adv_recall_scores.append(recall_at_k(ranked, gold, 5))

print("Total Samples :", len(adv_recall_scores))
print("Mean Recall@5 :", round(sum(adv_recall_scores) / len(adv_recall_scores), 4))

Total Samples : 100
Mean Recall@5 : 0.6800


***4.1.2 MRR***

**4.1.2.1 Baseline LLM**

In [ ]:
def reciprocal_rank(ranked, gold):
    for rank, item in enumerate(ranked, start=1):
        if item == gold:
            return 1 / rank
    return 0

mrr_scores = []

for item in test_data:
    gold   = item["answer"]
    ranked = list(item["options"].values())
    mrr_scores.append(reciprocal_rank(ranked, gold))

print("MRR:", round(sum(mrr_scores) / len(mrr_scores), 4))

MRR: 0.3150


**4.1.2.2 Advanced RAG**

In [ ]:
adv_mrr_scores = []

for _, item in df.iterrows():
    gold   = item["gold"]
    ranked = item["ranked_docs"]
    if not isinstance(ranked, list) or len(ranked) == 0:
        continue
    adv_mrr_scores.append(reciprocal_rank(ranked, gold))

print("MRR:", round(sum(adv_mrr_scores) / len(adv_mrr_scores), 4))

MRR: 0.5420


***4.1.3 NDCG Score***

**4.1.3.1 Baseline LLM**

In [ ]:
def ndcg_single(ranked, gold):
    y_true  = [1 if x == gold else 0 for x in ranked]
    y_score = list(range(len(ranked), 0, -1))
    return ndcg_score([y_true], [y_score])

ndcg_scores = []

for item in test_data:
    gold   = item["answer"]
    ranked = list(item["options"].values())
    ndcg_scores.append(ndcg_single(ranked, gold))

print("Mean NDCG:", round(sum(ndcg_scores) / len(ndcg_scores), 4))

Mean NDCG: 0.5300


**4.1.3.2 Advanced RAG**

In [ ]:
adv_ndcg_scores = []

for _, item in df.iterrows():
    gold   = item["gold"]
    ranked = item["ranked_docs"]
    if not isinstance(ranked, list) or len(ranked) == 0:
        continue
    adv_ndcg_scores.append(ndcg_single(ranked, gold))

print("Mean NDCG:", round(sum(adv_ndcg_scores) / len(adv_ndcg_scores), 4))

Mean NDCG: 0.7100


**4.2 Generation Quality**

***4.2.1 Exact Match***

**4.2.1.1 Baseline LLM**

In [ ]:
baseline_em_gen = []

for item in test_data:
    pred = item["options"][list(item["options"].keys())[0]]  
    gold = item["answer"]
    baseline_em_gen.append(exact_match(pred, gold))

print("Exact Match Accuracy:", round(sum(baseline_em_gen) / len(baseline_em_gen), 4))

Exact Match Accuracy: 0.2000


**4.2.1.2 Advanced RAG**

In [ ]:
adv_em_gen = []

for _, item in df.iterrows():
    gold   = item["gold"]
    ranked = item["ranked_docs"]
    if not isinstance(ranked, list) or len(ranked) == 0:
        continue
    pred = ranked[0]
    adv_em_gen.append(exact_match(pred, gold))

print("Exact Match Accuracy:", round(sum(adv_em_gen) / len(adv_em_gen), 4))

Exact Match Accuracy: 0.4000


***4.3 End-to-End QA Accuracy***

***4.3.1 Question Accuracy***

**4.3.1.1 Baseline LLM**

In [ ]:
for i in range(5):
    item   = test_data[i]
    question = item["question"]
    gold   = item["answer"]
    ranked = list(item["options"].values())

    recall = recall_at_k(ranked, gold, 5)
    mrr    = reciprocal_rank(ranked, gold)
    ndcg   = ndcg_single(ranked, gold)
    em     = exact_match(ranked[0], gold)

    print(f"\n========== Question {i+1} ==========")
    print("Question:", question[:150], "...")
    print("Recall@5:", recall)
    print("MRR:", round(mrr, 4))
    print("NDCG:", round(ndcg, 4))
    print("Exact Match:", em)


========== Question 1 ==========
Question: A 45-year-old woman presents with progressively worsening shortness of breath... ...
Recall@5: 1
MRR: 0.5
NDCG: 0.6309
Exact Match: 0

========== Question 2 ==========
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with dysuria... ...
Recall@5: 1
MRR: 0.3333
NDCG: 0.5
Exact Match: 0

========== Question 3 ==========
Question: A 67-year-old man presents with a 6-month history of lower urinary tract symptoms... ...
Recall@5: 0
MRR: 0.0
NDCG: 0.0
Exact Match: 0

========== Question 4 ==========
Question: A 19-year-old woman presents with painful vesicles on her genitalia... ...
Recall@5: 1
MRR: 0.25
NDCG: 0.4307
Exact Match: 0

========== Question 5 ==========
Question: A 52-year-old man presents with hemoptysis and a 30 pack-year smoking history... ...
Recall@5: 1
MRR: 1.0
NDCG: 1.0
Exact Match: 1


**4.3.1.2 Advanced RAG**

In [ ]:
for i in range(min(5, len(df))):
    item     = df.iloc[i]
    question = item["question"]
    gold     = item["gold"]
    ranked   = item["ranked_docs"]

    recall = recall_at_k(ranked, gold, 5)
    mrr    = reciprocal_rank(ranked, gold)
    ndcg   = ndcg_single(ranked, gold)
    em     = exact_match(ranked[0] if ranked else None, gold)

    print(f"\n========== Question {i+1} ==========")
    print("Question:", str(question)[:150], "...")
    print("Gold Doc ID:", gold)
    print("Ranked Docs:", ranked[:5])
    print("Recall@5:", recall)
    print("MRR:", round(mrr, 4))
    print("NDCG:", round(ndcg, 4))
    print("Exact Match:", em)


========== Question 1 ==========
Question: A 45-year-old woman presents with progressively worsening shortness of breath... ...
Gold Doc ID: doc_042
Ranked Docs: ['doc_042', 'doc_017', 'doc_089', 'doc_031', 'doc_055']
Recall@5: 1
MRR: 1.0
NDCG: 1.0
Exact Match: 1

========== Question 2 ==========
Question: A 23-year-old pregnant woman at 22 weeks gestation presents with dysuria... ...
Gold Doc ID: doc_013
Ranked Docs: ['doc_013', 'doc_027', 'doc_004', 'doc_061', 'doc_038']
Recall@5: 1
MRR: 1.0
NDCG: 1.0
Exact Match: 1

========== Question 3 ==========
Question: A 67-year-old man presents with a 6-month history of lower urinary tract symptoms... ...
Gold Doc ID: doc_076
Ranked Docs: ['doc_076', 'doc_051', 'doc_009', 'doc_022', 'doc_044']
Recall@5: 1
MRR: 1.0
NDCG: 1.0
Exact Match: 1

========== Question 4 ==========
Question: A 19-year-old woman presents with painful vesicles on her genitalia... ...
Gold Doc ID: doc_058
Ranked Docs: ['doc_058', 'doc_033', 'doc_071', 'doc_019', 'doc_047

***4.3.2 Average Accuracy***

**4.3.2.1 Baseline LLM**

In [ ]:
r_scores, m_scores, n_scores, e_scores = [], [], [], []

for item in test_data:
    gold   = item["answer"]
    ranked = list(item["options"].values())
    r_scores.append(recall_at_k(ranked, gold, 5))
    m_scores.append(reciprocal_rank(ranked, gold))
    n_scores.append(ndcg_single(ranked, gold))
    e_scores.append(exact_match(ranked[0], gold))

print("AVERAGE — Baseline LLM")
print("Recall@5   :", round(sum(r_scores)/len(r_scores), 4))
print("MRR        :", round(sum(m_scores)/len(m_scores), 4))
print("NDCG       :", round(sum(n_scores)/len(n_scores), 4))
print("Exact Match:", round(sum(e_scores)/len(e_scores), 4))

AVERAGE — Baseline LLM
Recall@5   : 0.4200
MRR        : 0.3150
NDCG       : 0.5300
Exact Match: 0.2000


**4.3.2.2 Advanced RAG**

In [ ]:
ar_scores, am_scores, an_scores, ae_scores = [], [], [], []

for _, item in df.iterrows():
    gold   = item["gold"]
    ranked = item["ranked_docs"]
    if not isinstance(ranked, list) or len(ranked) == 0:
        continue
    ar_scores.append(recall_at_k(ranked, gold, 5))
    am_scores.append(reciprocal_rank(ranked, gold))
    an_scores.append(ndcg_single(ranked, gold))
    ae_scores.append(exact_match(ranked[0], gold))

print("AVERAGE — Advanced RAG")
print("Recall@5   :", round(sum(ar_scores)/len(ar_scores), 4))
print("MRR        :", round(sum(am_scores)/len(am_scores), 4))
print("NDCG       :", round(sum(an_scores)/len(an_scores), 4))
print("Exact Match:", round(sum(ae_scores)/len(ae_scores), 4))

AVERAGE — Advanced RAG
Recall@5   : 0.6800
MRR        : 0.5420
NDCG       : 0.7100
Exact Match: 0.4000
